# Smart Crop Advisor (PAI101)
Run top to bottom. Part 1 prepares and analyses data, part 2 trains, evaluates and exports the models.

# 01 - Data preparation and analysis
Cleans both datasets, runs descriptive / exploratory / inferential analysis.
Run from anywhere: `python source/notebooks/01_prepare.py` (also opens as cells in PyCharm/VS Code).


In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
RAW, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed"
FIG = ROOT / "source" / "models" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
report = {}

## Crop recommendation data: check missing, duplicates, impossible values


In [2]:
crop = pd.read_csv(RAW / "crop_recommendation.csv")
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
report["crop_raw"] = {
    "rows": len(crop),
    "missing": int(crop.isna().sum().sum()),
    "duplicates": int(crop.duplicated().sum()),
    "ph_out_of_0_14": int((~crop.ph.between(0, 14)).sum()),
    "humidity_out_of_0_100": int((~crop.humidity.between(0, 100)).sum()),
    "negative_values": int((crop[FEATURES] < 0).any(axis=1).sum()),
}
crop = crop.drop_duplicates().dropna()
crop = crop[crop.ph.between(0, 14) & crop.humidity.between(0, 100) & (crop[FEATURES] >= 0).all(axis=1)]
crop.to_csv(OUT / "crop_clean.csv", index=False)
report["crop_clean_rows"] = len(crop)

## Descriptive + exploratory


In [3]:
report["describe"] = crop[FEATURES].describe().round(3).to_dict()
report["samples_per_crop"] = crop.label.value_counts().to_dict()
report["mean_per_crop"] = crop.groupby("label")[FEATURES].mean().round(2).to_dict("index")

sns.heatmap(crop[FEATURES].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Feature correlation"); plt.tight_layout(); plt.savefig(FIG / "correlation.png", dpi=120); plt.close()

crop[FEATURES].hist(bins=30, figsize=(11, 7))
plt.tight_layout(); plt.savefig(FIG / "histograms.png", dpi=120); plt.close()

plt.figure(figsize=(12, 5))
sns.boxplot(data=crop, x="label", y="rainfall")
plt.xticks(rotation=70); plt.tight_layout(); plt.savefig(FIG / "rainfall_by_crop.png", dpi=120); plt.close()

## Inferential: does rainfall differ between rice and chickpea? (Welch t-test + 95% CI) and across all crops (ANOVA)


In [4]:
rice, chick = (crop.loc[crop.label == c, "rainfall"] for c in ("rice", "chickpea"))
t = stats.ttest_ind(rice, chick, equal_var=False)
ci = t.confidence_interval(0.95)
report["ttest_rice_vs_chickpea_rainfall"] = {
    "mean_rice": round(rice.mean(), 2), "mean_chickpea": round(chick.mean(), 2),
    "t": round(t.statistic, 2), "p": float(t.pvalue),
    "ci95_mean_difference": [round(ci.low, 2), round(ci.high, 2)],
}
anova = stats.f_oneway(*(g.rainfall.values for _, g in crop.groupby("label")))
report["anova_rainfall_across_crops"] = {"F": round(anova.statistic, 1), "p": float(anova.pvalue)}

## Yield data


In [5]:
y = pd.read_csv(RAW / "yield_df.csv")
y = y.drop(columns=[c for c in y.columns if c.startswith("Unnamed")])  # leftover index column
report["yield_raw"] = {"rows": len(y), "missing": int(y.isna().sum().sum()), "duplicates": int(y.duplicated().sum())}
y = y.drop_duplicates().dropna()
y = y[(y["hg/ha_yield"] > 0) & (y.average_rain_fall_mm_per_year >= 0) & (y.pesticides_tonnes >= 0)]
y.to_csv(OUT / "yield_clean.csv", index=False)
report["yield_clean_rows"] = len(y)
report["yield_items"] = y.Item.value_counts().to_dict()
report["yield_describe"] = y.describe().round(2).to_dict()

sns.heatmap(y.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Yield data correlation"); plt.tight_layout(); plt.savefig(FIG / "yield_correlation.png", dpi=120); plt.close()

(ROOT / "source" / "models" / "eda.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps({k: report[k] for k in ("crop_raw", "crop_clean_rows", "yield_raw", "yield_clean_rows",
                                         "ttest_rice_vs_chickpea_rainfall", "anova_rainfall_across_crops")}, indent=2))

{
  "crop_raw": {
    "rows": 2200,
    "missing": 0,
    "duplicates": 0,
    "ph_out_of_0_14": 0,
    "humidity_out_of_0_100": 0,
    "negative_values": 0
  },
  "crop_clean_rows": 2200,
  "yield_raw": {
    "rows": 28242,
    "missing": 0,
    "duplicates": 2310
  },
  "yield_clean_rows": 25932,
  "ttest_rice_vs_chickpea_rainfall": {
    "mean_rice": 236.18,
    "mean_chickpea": 80.06,
    "t": 44.35,
    "p": 7.636899734027873e-72,
    "ci95_mean_difference": [
      149.15,
      163.1
    ]
  },
  "anova_rainfall_across_crops": {
    "F": 605.5,
    "p": 0.0
  }
}


# 01b - Synthetic crop dataset (weak and poor fields)
**This dataset is generated by us, not collected.** No public dataset we could find (Kaggle, Hugging Face, Mendeley)
contains weak or unsuitable fields: the Kaggle crop data only holds good fields for each crop.
So we add the missing side ourselves and call the result `synthetic_crop_dataset.csv`.

Rows (100 per crop, 2,200 total, same size as the real data):
* 40 `real_good`      - real rows of the crop (the strong conditions)
* 21 `other_crop`     - the conditions of each of the 21 other crops, so every crop x crop pair is covered once
* 30 `blend`          - the crop's own real row mixed 20-70% with another crop's conditions (marginal fields)
* 9  `one_reading_off`- a real row of the crop where 1-2 readings are replaced by another crop's values

`suitability` (0-1) is NOT random. It comes from one formula: how far the readings are from the crop's usual profile
(median and spread from the real data). Close = near 1, far = near 0. Under 0.30 means "unsuitable".
Run: `python source/notebooks/01b_synthetic.py` (after 01_prepare.py)


In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
SEED = 42
rng = np.random.default_rng(SEED)

real = pd.read_csv(PROC / "crop_clean.csv")
crops = sorted(real.label.unique())
lo, hi = real[FEATURES].min(), real[FEATURES].max()
# crop profile: median and spread (spread floored at 8% of the dataset range so very tight crops are not hypersensitive)
med = real.groupby("label")[FEATURES].median()
scale = real.groupby("label")[FEATURES].std().clip(lower=0.08 * (hi - lo), axis=1)


def suitability(crop, x):
    """1 = typical field for this crop, 0 = conditions of a very different crop. Mean deviation plus the worst reading."""
    z = (np.asarray(x, float) - med.loc[crop].values) / scale.loc[crop].values
    d2 = 0.5 * np.mean(z ** 2) + 0.5 * np.max(z ** 2) / 3.5
    return float(np.exp(-d2 / 2.6))


def tier(s):
    return "suitable" if s >= 0.70 else "marginal" if s >= 0.30 else "unsuitable"

In [7]:
rows = []


def add(crop, x, how, other=""):
    x = np.clip(x, lo.values, hi.values)
    s = round(suitability(crop, x), 3)
    rows.append({"crop": crop, **dict(zip(FEATURES, np.round(x, 2))), "suitability": s, "tier": tier(s),
                 "how_generated": how, "borrowed_from": other})


pick = lambda c: real[real.label == c][FEATURES].sample(1, random_state=int(rng.integers(1 << 30))).values[0]
for c in crops:
    own = real[real.label == c][FEATURES]
    for _, r in own.sample(40, random_state=SEED).iterrows():
        add(c, r.values, "real_good")
    for o in crops:
        if o != c:
            add(c, pick(o), "other_crop", o)  # every crop x other-crop pair appears exactly once
    for _ in range(30):
        o = rng.choice([k for k in crops if k != c]); a = rng.uniform(0.2, 0.7)
        add(c, (1 - a) * pick(c) + a * pick(o), "blend", o)
    for _ in range(9):
        o = rng.choice([k for k in crops if k != c]); x = pick(c); donor = pick(o)
        for j in rng.choice(7, size=rng.integers(1, 3), replace=False):
            x[j] = donor[j]
        add(c, x, "one_reading_off", o)

syn = pd.DataFrame(rows)
syn.to_csv(PROC / "synthetic_crop_dataset.csv", index=False)

## Sanity checks: are the labels sensible?


In [8]:
print(len(syn), "rows;", syn.crop.nunique(), "crops;", syn.groupby("crop").size().unique(), "per crop")
print(syn.tier.value_counts().to_string())
print(syn.groupby("how_generated").suitability.describe()[["count", "mean", "min", "50%", "max"]].round(2).to_string())
pairs = syn[syn.how_generated == "other_crop"].groupby(["crop", "borrowed_from"]).size()
assert len(pairs) == len(crops) * (len(crops) - 1), "every crop x other-crop pair must be present"
print("crop x other-crop pairs covered:", len(pairs), "of", len(crops) * (len(crops) - 1))
oc = syn[syn.how_generated == "other_crop"]
print(f"other crop's conditions scoring under 0.30: {(oc.suitability < .3).mean():.0%} (the rest are genuinely similar crops)")
print("real good fields scoring 0.70+:", f"{(syn[syn.how_generated == 'real_good'].suitability >= .7).mean():.0%}")

2200 rows; 22 crops; [100] per crop
tier
suitable      1050
unsuitable     740
marginal       410
                 count  mean   min   50%   max
how_generated                                 
blend            660.0  0.41  0.00  0.40  0.96
one_reading_off  198.0  0.50  0.00  0.61  0.96
other_crop       462.0  0.08  0.00  0.02  0.85
real_good        880.0  0.84  0.62  0.84  0.99
crop x other-crop pairs covered: 462 of 462
other crop's conditions scoring under 0.30: 92% (the rest are genuinely similar crops)
real good fields scoring 0.70+: 95%


In [9]:
import json  # crop profile (median, spread) is shared with training and the browser
(PROC / "crop_profile.json").write_text(json.dumps(
    {c: {"med": med.loc[c].round(4).tolist(), "scale": scale.loc[c].round(4).tolist()} for c in crops}))

3065

# 02 - Train, compare, explain, export
Modules 1-4, 6-8. Needs `01_prepare.py` to have run. Writes metrics + figures to source/models
and the browser model to frontend/data/model.json.


In [10]:
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, mean_absolute_error,
                             mean_squared_error, r2_score, silhouette_score, homogeneity_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore", message="X does not have valid feature names")
SEED = 42
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
PROC, MODELS = ROOT / "data" / "processed", ROOT / "source" / "models"
FIG = MODELS / "figures"
WEB = ROOT / "frontend" / "data"
WEB.mkdir(parents=True, exist_ok=True)
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
M = {}  # all metrics land here

## Module 1 - decision tree (depth chosen by cross-validation) vs random forest


In [11]:
crop = pd.read_csv(PROC / "crop_clean.csv")
X, y = crop[FEATURES], crop.label
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)  # stratified 80/20

cv = {d: cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=SEED), Xtr, ytr, cv=5).mean()
      for d in range(3, 13)}
depth = max(cv, key=cv.get)
M["tree_depth_cv"] = {str(d): round(s, 4) for d, s in cv.items()}
M["tree_depth_chosen"] = depth

t0 = time.time(); dt = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(Xtr, ytr); dt_time = time.time() - t0
t0 = time.time(); rf = RandomForestClassifier(n_estimators=30, max_depth=10, random_state=SEED).fit(Xtr, ytr); rf_time = time.time() - t0
M["decision_tree"] = {"accuracy": round(accuracy_score(yte, dt.predict(Xte)), 4), "train_seconds": round(dt_time, 3),
                      "train_accuracy": round(dt.score(Xtr, ytr), 4)}
M["random_forest"] = {"accuracy": round(accuracy_score(yte, rf.predict(Xte)), 4), "train_seconds": round(rf_time, 3),
                      "train_accuracy": round(rf.score(Xtr, ytr), 4)}
M["rf_per_crop"] = classification_report(yte, rf.predict(Xte), output_dict=True, zero_division=0)

ConfusionMatrixDisplay.from_predictions(yte, rf.predict(Xte), xticks_rotation=80, colorbar=False,
                                        ax=plt.subplots(figsize=(11, 10))[1])
plt.title("Random forest - confusion matrix (test set)"); plt.tight_layout()
plt.savefig(FIG / "confusion_rf.png", dpi=110); plt.close()

plt.figure(figsize=(18, 8))
plot_tree(dt, feature_names=FEATURES, class_names=list(dt.classes_), filled=True, max_depth=3, fontsize=7)
plt.savefig(FIG / "tree_top.png", dpi=110); plt.close()

imp = pd.Series(rf.feature_importances_, FEATURES).sort_values()
imp.plot.barh(title="Random forest feature importance"); plt.tight_layout()
plt.savefig(FIG / "feature_importance.png", dpi=110); plt.close()
M["feature_importance"] = imp.round(4).to_dict()

## Module 3 - neural network (scaler fit on the training set only)


In [12]:
scaler = StandardScaler().fit(Xtr)
t0 = time.time()
mlp = MLPClassifier((64, 32), activation="relu", max_iter=500, random_state=SEED).fit(scaler.transform(Xtr), ytr)
mlp_time = time.time() - t0
M["neural_net"] = {"accuracy": round(mlp.score(scaler.transform(Xte), yte), 4), "train_seconds": round(mlp_time, 3),
                   "train_accuracy": round(mlp.score(scaler.transform(Xtr), ytr), 4), "epochs": mlp.n_iter_}
plt.plot(mlp.loss_curve_); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("MLP loss curve")
plt.tight_layout(); plt.savefig(FIG / "mlp_loss.png", dpi=110); plt.close()

## Module 6 - SHAP on the random forest (offline; the web demo uses an occlusion explanation instead)


In [13]:
try:
    import shap
    sv = shap.TreeExplainer(rf).shap_values(Xte.iloc[:200])
    sv = np.abs(np.array(sv))
    mean_abs = sv.mean(axis=tuple(i for i in range(sv.ndim) if sv.shape[i] != len(FEATURES)))
    M["shap_mean_abs"] = dict(zip(FEATURES, np.round(mean_abs, 4).tolist()))
    pd.Series(mean_abs, FEATURES).sort_values().plot.barh(title="Mean |SHAP| (random forest)")
    plt.tight_layout(); plt.savefig(FIG / "shap_importance.png", dpi=110); plt.close()
except Exception as e:  # shap is optional
    M["shap_mean_abs"] = f"skipped: {e}"

C:\Users\super\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Module 4 - K-means zones (labels not used for fitting)


In [14]:
Xs_all = StandardScaler().fit(X)
Z = Xs_all.transform(X)
ks = range(2, 11)  # plotted for the elbow; k is picked from 4+ because k=2/3 gives zones too coarse to describe
inertia, sil = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=SEED).fit(Z)
    inertia.append(km.inertia_); sil.append(silhouette_score(Z, km.labels_))
k_best = max((k for k in ks if k >= 4), key=lambda k: sil[list(ks).index(k)])
km = KMeans(k_best, n_init=10, random_state=SEED).fit(Z)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ks, inertia, "o-"); ax[0].set_title("Elbow (inertia)")
ax[1].plot(ks, sil, "o-"); ax[1].set_title("Silhouette")
plt.tight_layout(); plt.savefig(FIG / "kmeans_k.png", dpi=110); plt.close()
P = PCA(2, random_state=SEED).fit_transform(Z)
plt.scatter(P[:, 0], P[:, 1], c=km.labels_, cmap="tab10", s=8); plt.title(f"K-means zones (k={k_best}) in PCA space")
plt.tight_layout(); plt.savefig(FIG / "kmeans_pca.png", dpi=110); plt.close()

WORDS = {"N": ("low nitrogen", "high nitrogen"), "P": ("low phosphorus", "high phosphorus"),
         "K": ("low potassium", "high potassium"), "temperature": ("cool", "hot"),
         "humidity": ("dry air", "humid"), "ph": ("acidic soil", "alkaline soil"),
         "rainfall": ("low rainfall", "high rainfall")}
zones = []
for i, c in enumerate(km.cluster_centers_):
    traits = [WORDS[f][c[j] > 0] for j, f in enumerate(FEATURES) if abs(c[j]) >= 0.5]
    crops = crop.label[km.labels_ == i].value_counts()
    zones.append({"id": i, "size": int((km.labels_ == i).sum()), "center": [round(float(v), 4) for v in c],
                  "description": ", ".join(traits) or "close to average conditions",
                  "top_crops": crops.head(4).index.tolist()})
M["kmeans"] = {"k": k_best, "silhouette_by_k": {str(k): round(s, 4) for k, s in zip(ks, sil)},
               "homogeneity_vs_crop": round(homogeneity_score(y, km.labels_), 4), "zones": zones}

C:\Users\super\AppData\Local\Temp\ipykernel_1904\443813805.py:24: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  traits = [WORDS[f][c[j] > 0] for j, f in enumerate(FEATURES) if abs(c[j]) >= 0.5]
C:\Users\super\AppData\Local\Temp\ipykernel_1904\443813805.py:24: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  traits = [WORDS[f][c[j] > 0] for j, f in enumerate(FEATURES) if abs(c[j]) >= 0.5]
C:\Users\super\AppData\Local\Temp\ipykernel_1904\443813805.py:24: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  traits = [WORDS[f][c[j] > 0] for j, f in enumerate(FEATURES) if abs(c[j]) >= 0.5]
C:\Users\super\AppData\Local\Temp\ipykernel_1904\443813805.py:24: DeprecationWarning: In future, it will be an error for 'np.bool' scalars to be interpreted as an index
  traits = [WORDS[f][c[j] > 0] for j, f in enumerate(FEATURES) if abs(c[

## Module 2 - yield regression (linear + ridge, one-hot crop type); random forest only as a reference


In [15]:
yd = pd.read_csv(PROC / "yield_clean.csv")
YF = ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp"]
Ytr, Yte = train_test_split(yd, test_size=0.2, stratify=yd.Item, random_state=SEED)
prep = ColumnTransformer([("item", OneHotEncoder(handle_unknown="ignore"), ["Item"]), ("num", "passthrough", YF)])
M["yield"] = {}
fitted = {}
for name, est in [("linear", LinearRegression()), ("ridge", Ridge(alpha=1.0)),
                  ("random_forest_reference", RandomForestRegressor(100, max_depth=14, n_jobs=-1, random_state=SEED))]:
    pipe = make_pipeline(prep, est).fit(Ytr[["Item"] + YF], Ytr["hg/ha_yield"])
    p = pipe.predict(Yte[["Item"] + YF])
    M["yield"][name] = {"MAE": round(mean_absolute_error(Yte["hg/ha_yield"], p)),
                        "RMSE": round(mean_squared_error(Yte["hg/ha_yield"], p) ** 0.5),
                        "R2": round(r2_score(Yte["hg/ha_yield"], p), 4)}
    fitted[name] = pipe
lin = fitted["linear"]
items = lin[0].named_transformers_["item"].categories_[0].tolist()
coef = lin[-1].coef_.tolist()

## Module 7 - Edge AI numbers; Module 8 - label-poisoning demo


In [16]:
def predict_ms(fn, n=200):
    t0 = time.perf_counter()
    for row in Xte.values[:n]:
        fn(row.reshape(1, -1))
    return round((time.perf_counter() - t0) / n * 1000, 3)

for name, mdl in [("decision_tree", dt), ("random_forest", rf), ("neural_net", mlp)]:
    f = MODELS / f"{name}.joblib"
    joblib.dump(mdl, f)
    M[name]["file_kb"] = round(f.stat().st_size / 1024, 1)
M["decision_tree"]["predict_ms"] = predict_ms(dt.predict)
M["random_forest"]["predict_ms"] = predict_ms(rf.predict)
M["neural_net"]["predict_ms"] = predict_ms(lambda r: mlp.predict(scaler.transform(r)))

rng = np.random.default_rng(SEED)
poison = {}
for frac in (0, 0.05, 0.10, 0.20, 0.30):
    yp = ytr.copy().values
    idx = rng.choice(len(yp), int(frac * len(yp)), replace=False)
    yp[idx] = rng.choice(np.unique(yp), len(idx))  # random wrong labels
    poison[str(frac)] = round(DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(Xtr, yp).score(Xte, yte), 4)
M["label_poisoning_tree_accuracy"] = poison

## Suitability model (trained on the synthetic crop dataset)
The classifiers above only learn from good fields. This model learns from the generated weak fields too.
Input = how far each reading is from the crop's usual profile (absolute distance, in spreads), output = suitability 0-1.
Gradient boosting with a **monotonic constraint**: a reading moving further from typical can never raise the score,
so improving a reading never lowers it (a plain neural network did not guarantee this).
Caveat for the report: the labels come from our own distance formula (01b_synthetic.py), so R2 below shows the model
reproduces that rule. The independent check is `argmax_matches_real_label`: on real held-out fields, does the
highest-suitability crop equal the true crop?


In [17]:
from sklearn.ensemble import HistGradientBoostingRegressor

syn = pd.read_csv(PROC / "synthetic_crop_dataset.csv")
profile = json.loads((PROC / "crop_profile.json").read_text())
zfeat = lambda c, x: np.abs(np.clip((np.asarray(x, float) - profile[c]["med"]) / profile[c]["scale"], -6, 6))
Zs = np.array([zfeat(c, r) for c, r in zip(syn.crop, syn[FEATURES].values)])
Str, Ste, str_, ste = train_test_split(Zs, syn.suitability.values, test_size=0.2, stratify=syn.crop, random_state=SEED)
sreg = HistGradientBoostingRegressor(monotonic_cst=[-1] * 7, max_iter=300, max_leaf_nodes=12, learning_rate=0.1,
                                     random_state=SEED).fit(Str, str_)
clip = lambda a: np.clip(a, 0, 1)
M["suitability"] = {"MAE": round(mean_absolute_error(ste, clip(sreg.predict(Ste))), 4),
                    "R2": round(r2_score(ste, clip(sreg.predict(Ste))), 4), "rows": len(syn)}
bands = lambda a: pd.cut(a, [-1, .3, .7, 2], labels=["unsuitable", "marginal", "suitable"])
M["suitability"]["tier_accuracy"] = round(float((bands(ste) == bands(clip(sreg.predict(Ste)))).mean()), 4)
M["suitability"]["unsuitable_predicted_under_30pct"] = round(float((clip(sreg.predict(Ste))[ste < .3] < .3).mean()), 4)
# independent check on REAL held-out fields (not generated): highest-suitability crop vs the true crop
cls = rf.classes_.tolist()
suit_all = lambda x: [float(clip(sreg.predict([zfeat(c, x)]))[0]) for c in cls]
hit = [cls[int(np.argmax(suit_all(x)))] == t for x, t in zip(Xte.values, yte)]
M["suitability"]["argmax_matches_real_label"] = round(float(np.mean(hit)), 4)
print(M["suitability"])

{'MAE': 0.0506, 'R2': 0.9438, 'rows': 2200, 'tier_accuracy': 0.9091, 'unsuitable_predicted_under_30pct': 0.8581, 'argmax_matches_real_label': 0.975}


## Export for the browser (Edge AI: everything runs client-side)


In [18]:
def tree_json(est):
    t = est.tree_
    val = t.value[:, 0, :]
    val = val / val.sum(1, keepdims=True)
    v = [None if t.children_left[i] != -1 else
         [x for c, p in enumerate(val[i]) if p >= 0.005 for x in (c, round(float(p), 3))]
         for i in range(t.node_count)]
    return {"f": t.feature.tolist(), "t": [round(float(x), 6) for x in t.threshold],
            "l": t.children_left.tolist(), "r": t.children_right.tolist(), "v": v}

stats = {c: {f: [round(float(np.percentile(g[f], q)), 2) for q in (10, 50, 90)] for f in FEATURES}
         for c, g in crop.groupby("label")}  # [p10, median, p90] per crop - the rule engine's thresholds
yield_for_crop = {"maize": "Maize", "rice": "Rice, paddy"}
assert all(v in items for v in yield_for_crop.values())

model = {
    "features": FEATURES, "classes": rf.classes_.tolist(),
    "ranges": {f: [float(X[f].min()), float(X[f].max())] for f in FEATURES},
    "crop_stats": stats,
    "tree": tree_json(dt), "forest": [tree_json(e) for e in rf.estimators_],
    "importance": M["feature_importance"],
    "mlp": {"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
            "W": [np.round(w, 6).tolist() for w in mlp.coefs_], "b": [np.round(b, 6).tolist() for b in mlp.intercepts_]},
    "kmeans": {"mean": Xs_all.mean_.tolist(), "scale": Xs_all.scale_.tolist(), "zones": zones},
    "yield": {"items": items, "intercept": float(lin[-1].intercept_), "item_coef": coef[:len(items)],
              "num_features": YF, "num_coef": coef[len(items):],
              "ranges": {f: [float(yd[f].min()), float(yd[f].max())] for f in YF},
              "median_pesticides": float(yd.pesticides_tonnes.median()),
              "for_crop": yield_for_crop, "metrics": M["yield"]["linear"]},
    "suit": {"profile": profile, "base": float(sreg._baseline_prediction.ravel()[0]), "metrics": M["suitability"],
             "trees": [{"f": n["feature_idx"].tolist(), "t": [round(float(v), 4) for v in n["num_threshold"]],
                        "l": n["left"].tolist(), "r": n["right"].tolist(), "v": [round(float(v), 5) for v in n["value"]],
                        "leaf": n["is_leaf"].astype(int).tolist()} for n in (p[0].nodes for p in sreg._predictors)]},
    "metrics": {k: M[k] for k in ("decision_tree", "random_forest", "neural_net")},
}
out = WEB / "model.json"
out.write_text(json.dumps(model, separators=(",", ":")))
M["web_model_kb"] = round(out.stat().st_size / 1024, 1)

# parity cases: the JS port must reproduce these (see source/tests/parity.mjs)
cases = Xte.iloc[:150]
(ROOT / "source" / "tests").mkdir(exist_ok=True)
(ROOT / "source" / "tests" / "parity_cases.json").write_text(json.dumps({
    "x": cases.values.tolist(), "tree": dt.predict(cases).tolist(), "forest": rf.predict(cases).tolist(),
    "mlp": mlp.predict(scaler.transform(cases)).tolist(),
    "zone": km.predict(Xs_all.transform(cases)).tolist(),
    "suit": [[c, [round(float(v), 4) for v in x], round(float(clip(sreg.predict([zfeat(c, x)]))[0]), 4)]
             for c, x in zip(["rice", "apple", "lentil", "mothbeans", "coffee"] * 6, Xte.iloc[:30].values)]}))

(MODELS / "metrics.json").write_text(json.dumps(M, indent=2, default=str))
print(json.dumps({k: M[k] for k in ("tree_depth_chosen", "decision_tree", "random_forest", "neural_net", "yield",
                                    "label_poisoning_tree_accuracy", "web_model_kb")}, indent=2))
print("kmeans k =", k_best, "homogeneity", M["kmeans"]["homogeneity_vs_crop"])
for z in zones: print(z["id"], z["size"], z["description"], z["top_crops"])

{
  "tree_depth_chosen": 11,
  "decision_tree": {
    "accuracy": 0.9773,
    "train_seconds": 0.013,
    "train_accuracy": 0.9898,
    "file_kb": 15.7,
    "predict_ms": 0.081
  },
  "random_forest": {
    "accuracy": 0.9909,
    "train_seconds": 0.092,
    "train_accuracy": 0.9966,
    "file_kb": 773.1,
    "predict_ms": 1.145
  },
  "neural_net": {
    "accuracy": 0.9909,
    "train_seconds": 1.984,
    "train_accuracy": 0.9983,
    "epochs": 408,
    "file_kb": 121.1,
    "predict_ms": 0.137
  },
  "yield": {
    "linear": {
      "MAE": 32040,
      "RMSE": 51004,
      "R2": 0.6376
    },
    "ridge": {
      "MAE": 32037,
      "RMSE": 51004,
      "R2": 0.6376
    },
    "random_forest_reference": {
      "MAE": 8034,
      "RMSE": 15349,
      "R2": 0.9672
    }
  },
  "label_poisoning_tree_accuracy": {
    "0": 0.9773,
    "0.05": 0.9545,
    "0.1": 0.925,
    "0.2": 0.8114,
    "0.3": 0.825
  },
  "web_model_kb": 194.5
}
kmeans k = 10 homogeneity 0.6404
0 79 hot, humid, high